# CNN LeNet-5 – MNIST / Fashion MNIST / Medical MNIST
**Mục tiêu:**
1. Thiết kế và train LeNet-5 trên 3 dataset
2. Phân tích kết quả (confusion matrix, wrong predictions)
3. So sánh Baseline vs Improved và đề xuất cải tiến

## 0. Cài đặt & Import

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/working')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Kiểm tra dữ liệu Medical MNIST

In [ ]:
import os
medical_path = '/kaggle/input/datasets/andrewmvd/medical-mnist'
print(os.listdir(medical_path))

# Kiểm tra số lượng ảnh mỗi lớp
for cls in sorted(os.listdir(medical_path)):
    cls_path = os.path.join(medical_path, cls)
    if os.path.isdir(cls_path):
        print(f'  {cls}: {len(os.listdir(cls_path))} images')

## 2. Load Dataset

In [ ]:
from config  import DEVICE, BATCH_SIZE, EPOCHS
from dataset import get_all_datasets

all_ds = get_all_datasets(batch_size=BATCH_SIZE)

for name, info in all_ds.items():
    n_train = len(info['train'].dataset)
    n_test  = len(info['test'].dataset)
    print(f'{name:10s}: train={n_train}, test={n_test}, classes={info["num_classes"]}')
    print(f'  Class names: {info["classes"]}')

## 3. Visualise mẫu dữ liệu

In [ ]:
def show_samples(loader, class_names, title, n=10):
    images, labels = next(iter(loader))
    fig, axes = plt.subplots(1, n, figsize=(16, 2))
    fig.suptitle(title, fontsize=13)
    for i in range(n):
        axes[i].imshow(images[i].squeeze(), cmap='gray')
        axes[i].set_title(class_names[labels[i]], fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

show_samples(all_ds['handwritten']['train'], all_ds['handwritten']['classes'], 'Handwritten MNIST')
show_samples(all_ds['fashion']['train'], all_ds['fashion']['classes'], 'Fashion MNIST')
show_samples(all_ds['medical']['train'], all_ds['medical']['classes'], 'Medical MNIST')

## 4. Kiến trúc LeNet-5

In [ ]:
from model import LeNet5, LeNet5Improved

# In tóm tắt kiến trúc
model_demo = LeNet5(num_classes=10)
print('=== LeNet-5 Baseline ===')
print(model_demo)
print(f'Total params: {sum(p.numel() for p in model_demo.parameters()):,}')

model_imp = LeNet5Improved(num_classes=10)
print('\n=== LeNet-5 Improved ===')
print(model_imp)
print(f'Total params: {sum(p.numel() for p in model_imp.parameters()):,}')

## 5. Train – Baseline LeNet-5

In [ ]:
from train import train_all, print_summary

results_baseline = train_all(all_ds, improved=False, epochs=EPOCHS)
print_summary(results_baseline)

## 6. Train – Improved LeNet-5

In [ ]:
results_improved = train_all(all_ds, improved=True, epochs=EPOCHS)
print_summary(results_improved)

## 7. So sánh Loss & Accuracy Curve

In [ ]:
def plot_history(results_base, results_imp, ds_name):
    h_b = results_base[ds_name]['history']
    h_i = results_imp[ds_name]['history']
    epochs = range(1, len(h_b['loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{ds_name.upper()} – Training curves', fontsize=13)

    # Loss
    axes[0].plot(epochs, h_b['loss'], label='Baseline')
    axes[0].plot(epochs, h_i['loss'], label='Improved')
    axes[0].set_title('Loss'); axes[0].legend()

    # Train Acc
    axes[1].plot(epochs, h_b['train_acc'], label='Baseline')
    axes[1].plot(epochs, h_i['train_acc'], label='Improved')
    axes[1].set_title('Train Accuracy'); axes[1].legend()

    # Test Acc
    axes[2].plot(epochs, h_b['test_acc'], label='Baseline')
    axes[2].plot(epochs, h_i['test_acc'], label='Improved')
    axes[2].set_title('Test Accuracy'); axes[2].legend()

    plt.tight_layout()
    plt.show()

for ds in ['handwritten', 'fashion', 'medical']:
    plot_history(results_baseline, results_improved, ds)

## 8. Confusion Matrix

In [ ]:
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels, all_images = [], [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            preds = model(X).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_images.extend(X.cpu())
    return all_preds, all_labels, all_images


def plot_confusion_matrix(preds, labels, class_names, title):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# Vẽ confusion matrix cho Improved model
for ds_name in ['mnist', 'fashion', 'medical']:
    model  = results_improved[ds_name]['model']
    loader = all_ds[ds_name]['test']
    classes = all_ds[ds_name]['classes']

    preds, labels, images = get_predictions(model, loader)
    plot_confusion_matrix(preds, labels, classes, f'{ds_name.upper()} – Confusion Matrix (Improved)')
    print(classification_report(labels, preds, target_names=classes))

    # Lưu lại để dùng tiếp
    all_ds[ds_name]['preds']  = preds
    all_ds[ds_name]['labels'] = labels
    all_ds[ds_name]['images'] = images

## 9. Xem ảnh dự đoán sai

In [ ]:
def show_wrong_predictions(images, preds, labels, class_names, title, num_images=9):
    wrong = [i for i in range(len(preds)) if preds[i] != labels[i]]
    if not wrong:
        print('No wrong predictions!')
        return

    n = min(num_images, len(wrong))
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    fig.suptitle(f'{title} - Wrong predictions ({len(wrong)} total)', fontsize=12)

    for i, ax in enumerate(axes.flatten()):
        if i >= n:
            ax.axis('off'); continue
        idx = wrong[i]
        ax.imshow(images[idx].squeeze(), cmap='gray')
        ax.set_title(f'True: {class_names[labels[idx]]}\nPred: {class_names[preds[idx]]}', fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


for ds_name in ['mnist', 'fashion', 'medical']:
    show_wrong_predictions(
        all_ds[ds_name]['images'],
        all_ds[ds_name]['preds'],
        all_ds[ds_name]['labels'],
        all_ds[ds_name]['classes'],
        ds_name.upper()
    )

## 10. Tổng hợp kết quả & Phân tích

### Bảng kết quả


In [ ]:
import pandas as pd

rows = []
for ds_name in ['mnist', 'fashion', 'medical']:
    best_base = max(results_baseline[ds_name]['history']['test_acc'])
    best_imp  = max(results_improved[ds_name]['history']['test_acc'])
    rows.append({
        'Dataset'   : ds_name.upper(),
        'Baseline'  : f'{best_base:.4f}',
        'Improved'  : f'{best_imp:.4f}',
        'Delta'     : f'{best_imp - best_base:+.4f}',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 11. Lưu model


In [ ]:
import os
save_dir = '/kaggle/working/checkpoints'
os.makedirs(save_dir, exist_ok=True)

for ds_name in ['handwritten', 'fashion', 'medical']:
    path = f'{save_dir}/lenet5_improved_{ds_name}.pth'
    torch.save(results_improved[ds_name]['model'].state_dict(), path)
    print(f'Saved: {path}')